In [ ]:
import pandas as pd
from selenium import webdriver
from selenium_stealth import stealth
from bs4 import BeautifulSoup
import time

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
driver = webdriver.Chrome(options=options)

stealth(driver,
        languages=["en-US", "en"],
        vendor="Google Inc.",
        platform="Win32",
        webgl_vendor="Intel Inc.",
        renderer="Intel Iris OpenGL Engine",
        fix_hairline=True)

all_dfs = [] # Sab pages ka data yahan jama hoga

for j in range(1,50): # 1 se 50 pages tak scrape karna hai
    url = f"https://www.olx.com.pk/karachi_g4060695/property-for-sale_c2?page={j}"
    print(f"Scraping Page {j}...")
    driver.get(url)
    time.sleep(8)
    
    driver.execute_script("window.scrollTo(0, 500);") 
    time.sleep(2)
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    properties = soup.find_all('article', class_="_65e508e1")
    
    page_data = []
    for i in properties:
        try:
            p = i.find('span', class_="_63bc9c97").text if i.find('span', class_="_63bc9c97") else "0"
            a = i.find("span", attrs={"aria-label": "Area"}).text if i.find("span", attrs={"aria-label": "Area"}) else "N/A"
            bed = i.find("span", attrs={"aria-label": "Beds"}).text if i.find("span", attrs={"aria-label": "Beds"}) else "0"
            bath = i.find("span", attrs={"aria-label": "Bathrooms"}).text if i.find("span", attrs={"aria-label": "Bathrooms"}) else "0"
            locat=i.find('span', class_='_97d73a54').text if i.find('span', class_='_97d73a54') else "N/A"
            page_data.append({
                'price': p,
                'area': a,
                'bedrooms': bed,
                'bathrooms': bath,
                'location': locat
            })
        except Exception as e:
            continue
            
    df = pd.DataFrame(page_data)
    all_dfs.append(df) 

if all_dfs:
    final = pd.concat(all_dfs, ignore_index=True)
    final.to_csv("final_cleaned.csv", index=False)
else:
    print("some error occured")

driver.quit()

ModuleNotFoundError: No module named 'selenium'